In [ ]:
# 1. CÀI ĐẶT
!pip uninstall -y transformers adapters huggingface_hub accelerate peft sentence-transformers
!pip install -q huggingface_hub==0.23.0 transformers==4.39.3 adapters==0.2.1 accelerate==0.27.2 datasets kaggle

import os, shutil, json, torch, nltk, numpy as np
from google.colab import drive
from transformers import AutoTokenizer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset
from adapters import AutoAdapterModel, AdapterTrainer

os.environ["WANDB_DISABLED"] = "true"

try:
    torch.serialization.add_safe_globals([np.core.multiarray._reconstruct, np.ndarray, np.dtype])
except AttributeError: pass

if not hasattr(torch, 'original_load_func'): torch.original_load_func = torch.load
def safe_load_override(*args, **kwargs):
    if 'weights_only' in kwargs: del kwargs['weights_only']
    return torch.original_load_func(*args, weights_only=False, **kwargs)
torch.load = safe_load_override

# 2. THIẾT LẬP DỮ LIỆU
os.environ['KAGGLE_USERNAME'] = "phankhaclap"
os.environ['KAGGLE_KEY'] = "0ba946628cb1f5acb76ecd357f590e95"
drive.mount('/content/drive')

FINAL_SAVE_PATH = "/content/drive/My Drive/SLM_Research/BART_Base_Spider_Houlsby"
CHECKPOINT_DIR = "/content/drive/My Drive/SLM_Research/BART_Base_Spider_Houlsby/checkpoints"

if not os.path.exists('spider_data'):
    !kaggle datasets download -d jeromeblanchet/yale-universitys-spider-10-nlp-dataset
    !unzip -q yale-universitys-spider-10-nlp-dataset.zip -d temp_spider
    shutil.move("temp_spider/spider", "spider_data")
    shutil.rmtree('temp_spider')
    !wget -q https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py
    !wget -q https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py
    nltk.download('punkt')
    nltk.download('punkt_tab')

with open('spider_data/tables.json', 'r') as f: table_data = json.load(f)
schema_map = {db['db_id']: db for db in table_data}

def get_crp_schema(db_id):
    if db_id not in schema_map: return ""
    db = schema_map[db_id]
    ddls = []
    for i, table_name in enumerate(db['table_names_original']):
        cols = [f"{c[1]} {db['column_types'][j]}" for j, c in enumerate(db['column_names_original']) if c[0] == i]
        ddls.append(f"CREATE TABLE {table_name} ({', '.join(cols)});")
    return " ".join(ddls)

MODEL_NAME = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_fn(examples):
    inputs = [f"translate to SQL: {q} | Schema: {get_crp_schema(d)}" for q, d in zip(examples['question'], examples['db_id'])]
    model_inputs = tokenizer(inputs, max_length=512, padding="max_length", truncation=True)

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples['query'], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def load_ds(path):
    with open(path, 'r', encoding='utf-8') as f: data = json.load(f)
    return Dataset.from_dict({"question": [x["question"] for x in data], "query": [x["query"] for x in data], "db_id": [x["db_id"] for x in data]})

train_ds = load_ds("spider_data/train_spider.json").map(preprocess_fn, batched=True)
val_ds = load_ds("spider_data/dev.json").map(preprocess_fn, batched=True)

# 3. KHỞI TẠO BART + HOULSBY ADAPTER

model = AutoAdapterModel.from_pretrained(MODEL_NAME)
adapter_name = "spider_bart"

model.add_adapter(adapter_name, config="seq_bn")
model.add_seq2seq_lm_head(adapter_name)
model.set_active_adapters(adapter_name)
model.train_adapter(adapter_name)


data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)


# 4. HUẤN LUYỆN

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=15,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    weight_decay=0.01,
    fp16= False,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    report_to="none",
    logging_steps=50,
    load_best_model_at_end=False
)

trainer = AdapterTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator
)

trainer.train()

# 5. Lưu và Đánh giá
model.save_adapter(FINAL_SAVE_PATH, adapter_name)
tokenizer.save_pretrained(FINAL_SAVE_PATH)

model.eval()
predictions, gold_lines = [], []
for item in val_ds:
    input_text = f"translate to SQL: {item['question']} | Schema: {get_crp_schema(item['db_id'])}"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        output = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
    predictions.append(tokenizer.decode(output[0], skip_special_tokens=True) + "\n")
    gold_lines.append(f"{item['query']}\t{item['db_id']}\n")

with open('pred.txt', 'w') as f: f.writelines(predictions)
with open('gold.txt', 'w') as f: f.writelines(gold_lines)

!python evaluation.py --gold gold.txt --pred pred.txt --db spider_data/database --table spider_data/tables.json --etype all

Found existing installation: transformers 4.39.3
Uninstalling transformers-4.39.3:
  Successfully uninstalled transformers-4.39.3
Found existing installation: adapters 0.2.1
Uninstalling adapters-0.2.1:
  Successfully uninstalled adapters-0.2.1
Found existing installation: huggingface-hub 0.23.0
Uninstalling huggingface-hub-0.23.0:
  Successfully uninstalled huggingface-hub-0.23.0
Found existing installation: accelerate 0.27.2
Uninstalling accelerate-0.27.2:
  Successfully uninstalled accelerate-0.27.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.36.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.23.0 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.23.0 which is incompatible.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount(

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:3935: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
0,1.417500,0.981031
1,1.019400,0.785882
2,0.843700,0.708438
4,0.685800,0.636708
5,0.640600,0.609581
6,0.609300,0.586042
8,0.556400,0.567913
9,0.531400,0.561906
10,0.521500,0.554810
12,0.486000,0.549710


easy pred: SELECT sum(*) FROM singer
easy gold: SELECT count(*) FROM singer

medium pred: SELECT Name,  Age FROM singer ORDER BY Age DESC
medium gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC

medium pred: SELECT Name,  Country,  Age FROM singer ORDER BY Age ASC
medium gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC

eval_err_num:1
medium pred: SELECT avg(age),  min(age); FROM singer WHERE country  =  "France"
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

eval_err_num:2
medium pred: SELECT avg(age),  min(age); FROM singer GROUP BY singer_id
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

eval_err_num:3
medium pred: SELECT Song_Name,  YEAR FROM singer ORDER BY Age DESC LIMIT 1
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

medium pred: SELECT Name,  Song_release_year FROM singer ORDER BY Age DESC LIMIT 1
medium gold: SELEC

In [1]:
!pip uninstall -y transformers adapters huggingface_hub accelerate peft sentence-transformers
!pip install -q huggingface_hub==0.23.0 transformers==4.39.3 adapters==0.2.1 accelerate==0.27.2 datasets kaggle

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: huggingface_hub 1.5.0
Uninstalling huggingface_hub-1.5.0:
  Successfully uninstalled huggingface_hub-1.5.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: sentence-transformers 5.2.3
Uninstalling sentence-transformers-5.2.3:
  Successfully uninstalled sentence-transformers-5.2.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:
# ĐÁNH GIÁ HIỆU NĂNG CHO MÔ HÌNH ADAPTERS (BART-Base)

import torch
import time
import numpy as np
import psutil
import os
import gc
from transformers import AutoTokenizer
from adapters import AutoAdapterModel
from google.colab import drive

# 1. Kết nối Drive và dọn dẹp bộ nhớ
drive.mount('/content/drive')

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

BASE_MODEL_NAME = "facebook/bart-base"

ADAPTER_SAVE_PATH = "/content/drive/My Drive/BART_Base_Spider_Houlsby"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. BỘ DÒ TÌM ĐƯỜNG DẪN ADAPTER TỰ ĐỘNG
print("Đang dò tìm vị trí chính xác của file Adapter...")
actual_adapter_path = None

if os.path.exists(ADAPTER_SAVE_PATH):
    for root, dirs, files in os.walk(ADAPTER_SAVE_PATH):
        if "adapter_config.json" in files:
            actual_adapter_path = root
            if root == ADAPTER_SAVE_PATH:
                break

if not actual_adapter_path:
    raise FileNotFoundError(f" Không tìm thấy 'adapter_config.json'. Hãy kiểm tra lại đường dẫn!")

print(f"✅ Đã tìm thấy Adapter tại: {actual_adapter_path}")

# 3. Tải Tokenizer và Mô hình
print("Đang tải Tokenizer và Base Model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
model = AutoAdapterModel.from_pretrained(BASE_MODEL_NAME)

model.load_adapter(actual_adapter_path, set_active=True)
model.to(device)
model.eval()
print("✅ Tải thành công mô hình BART-Base với Adapters!")

# 4. Dữ liệu đầu vào
input_text = "translate to SQL: How many students are there? | Schema: CREATE TABLE student(id int, name text);"
inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)

# 5. Warmup GPU
print("Đang warmup GPU...")
for _ in range(10):
    with torch.no_grad():
        model.generate(**inputs, max_length=128)

# 6. Đo Latency (Độ trễ)
print("Đang đo lường độ trễ (100 lần chạy)...")
latencies = []
for _ in range(100):
    start = time.time()
    with torch.no_grad():
        model.generate(**inputs, max_length=128)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    latencies.append(time.time() - start)

avg_latency = np.mean(latencies)
std_latency = np.std(latencies)

# 7. Thông lượng (Throughput)
throughput = 1 / avg_latency

# 8. Đo VRAM và RAM
gpu_memory = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else None
process = psutil.Process(os.getpid())
ram_usage = process.memory_info().rss / 1024**2
total_params = sum(p.numel() for p in model.parameters())

# 9. In kết quả
print("\n" + "="*45)
print("   ĐÁNH GIÁ HIỆU NĂNG (BART-BASE + ADAPTERS)  ")
print("="*45)
print(f"Độ trễ TB (Latency):            {avg_latency*1000:.2f} ms")
print(f"Độ lệch chuẩn (Std):            {std_latency*1000:.2f} ms")
print(f"Thông lượng (Throughput BS=1):  {throughput:.2f} samples/sec")
if gpu_memory:
    print(f"VRAM tối đa đã dùng (Peak):     {gpu_memory:.2f} MB")
print(f"RAM CPU đang sử dụng:           {ram_usage:.2f} MB")
print(f"Tổng số tham số mô hình:        {total_params:,}")
print("="*45)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đang dò tìm vị trí chính xác của file Adapter...
✅ Đã tìm thấy Adapter tại: /content/drive/My Drive/BART_Base_Spider_Houlsby/checkpoints/checkpoint-3270/spider_bart
Đang tải Tokenizer và Base Model...
✅ Tải thành công mô hình BART-Base với Adapters!
Đang warmup GPU...
Đang đo lường độ trễ (100 lần chạy)...

   ĐÁNH GIÁ HIỆU NĂNG (BART-BASE + ADAPTERS)  
Độ trễ TB (Latency):            322.20 ms
Độ lệch chuẩn (Std):            82.19 ms
Thông lượng (Throughput BS=1):  3.10 samples/sec
VRAM tối đa đã dùng (Peak):     1115.26 MB
RAM CPU đang sử dụng:           2108.27 MB
Tổng số tham số mô hình:        140,314,944
